# Notebook 03 — Making Text2Cypher robust

Zero-shot works for easy questions. In this notebook we add four layers, each one a classic technique:

| Layer | Technique | Fixes |
|---|---|---|
| 1 | **Read-only guard** + EXPLAIN | safety, syntax errors before execution |
| 2 | **Entity linking** (skills, people, jobs, cities) | aliases, ambiguous names, IDs |
| 3 | **Dynamic few-shot examples** (embedding similarity) | multi-hop patterns, business rules |
| 4 | **Self-correction loop** | errors are fed back to the model |

At the end, all layers are combined in a `Text2Cypher` class (the same class is in `t2c.py` and is reused in notebooks 04–05).

In [1]:
import os, sys, json, re, time, math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --- Connection settings (override with environment variables if you prefer) ---
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")     # <- change me
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None          # None = default database

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL       = os.getenv("T2C_MODEL", "qwen2.5-coder:7b")      # any chat model you pulled with `ollama pull`
EMBED_MODEL = os.getenv("T2C_EMBED_MODEL", "nomic-embed-text")

print("Neo4j :", NEO4J_URI, "| Ollama:", OLLAMA_HOST, "| model:", MODEL)

Neo4j : bolt://localhost:7687 | Ollama: http://localhost:11434 | model: qwen2.5:7b


In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)

from t2c import Neo4jGraph, extract_schema, schema_to_text

graph = Neo4jGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
schema_text = schema_to_text(extract_schema(graph))
print("Connected. Schema description:", len(schema_text), "characters")

from t2c import OllamaLLM, build_system_prompt, extract_cypher

llm = OllamaLLM(MODEL, host=OLLAMA_HOST, temperature=0.0, num_ctx=8192)
system_prompt = build_system_prompt(schema_text)

Connected. Schema description: 5371 characters


## Layer 1: Safety

### Step 1: A read-only guard
**What it does:** `check_read_only` rejects a generated query if it
* contains a write/admin keyword (`CREATE`, `MERGE`, `SET`, `DELETE`, `REMOVE`, `DROP`, `FOREACH`, `LOAD CSV`, `CALL`),
* contains more than one statement (`;`),
* does not `RETURN` anything.

String literals and comments are blanked first, so a skill called `'SET theory'` does not trigger a false alarm.

**Why `CALL` is blocked:** procedures can do anything (`CALL dbms.…`, APOC file access). We lose `CALL { subquery }` too — an acceptable trade-off for a teaching system.

In [4]:
class UnsafeQueryError(ValueError):
    pass

WRITE_PATTERN = re.compile(
    '\\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|FOREACH|LOAD\\s+CSV|CALL)\\b', re.IGNORECASE)
STRING_LITERAL = re.compile('\'(?:[^\'\\\\]|\\\\.)*\'|\\"(?:[^\\"\\\\]|\\\\.)*\\"')

def check_read_only(cypher: str) -> str:
    """Reject anything that is not a single read-only query. Returns the query if OK."""
    code = STRING_LITERAL.sub("''", cypher)            # ignore keywords inside string literals
    code = re.sub(r"//.*", "", code)                    # ignore comments
    if ";" in code.strip().rstrip(";"):
        raise UnsafeQueryError("Only one statement is allowed.")
    found = WRITE_PATTERN.search(code)
    if found:
        raise UnsafeQueryError(f"Forbidden clause: {found.group(1).upper()}")
    if not re.search(r"\bRETURN\b", code, re.IGNORECASE):
        raise UnsafeQueryError("The query must RETURN something.")
    return cypher

### Step 2: Test the guard
**What it does:** runs the guard on safe and malicious queries and shows the decision. Encourage students to try to bypass it (e.g. lowercase `detach delete`, comments, string tricks).

In [5]:
tests = [
    "MATCH (j:Job {status: 'OPEN'}) RETURN j.title",
    "MATCH (n) DETACH DELETE n",
    "match (c:Candidate) set c.name = 'x' return c",
    "MATCH (s:Skill {name: 'SET theory'}) RETURN s",
    "MATCH (n) RETURN n; MATCH (m) DELETE m",
    "CALL dbms.components() YIELD name RETURN name",
]
for q in tests:
    try:
        check_read_only(q); verdict = "✅ allowed"
    except UnsafeQueryError as e:
        verdict = f"⛔ {e}"
    print(f"{verdict:40s} {q}")

✅ allowed                                MATCH (j:Job {status: 'OPEN'}) RETURN j.title
⛔ Forbidden clause: DETACH               MATCH (n) DETACH DELETE n
⛔ Forbidden clause: SET                  match (c:Candidate) set c.name = 'x' return c
✅ allowed                                MATCH (s:Skill {name: 'SET theory'}) RETURN s
⛔ Only one statement is allowed.         MATCH (n) RETURN n; MATCH (m) DELETE m
⛔ Forbidden clause: CALL                 CALL dbms.components() YIELD name RETURN name


### Step 3: Defense in depth: the database also says no
**What it does:** tries to run a `CREATE` through `graph.run_read` (bypassing our regex on purpose).
**Expected:** Neo4j raises an error because the transaction is in READ mode. A regex can always be bypassed; the access mode cannot. In production you would also use a **database user with the `reader` role**.

In [6]:
try:
    graph.run_read("CREATE (:Hacked {at: datetime()}) RETURN 1")
    print("Write succeeded?! Check your setup.")
except Exception as exc:
    print("Blocked by Neo4j:", type(exc).__name__, str(exc)[:200])

Blocked by Neo4j: ClientError {neo4j_code: Neo.ClientError.Statement.AccessMode} {message: Writing in read access mode not allowed. Attempted write to euromedtp} {gql_status: 50N42} {gql_status_description: error: general processi


### Step 4 : Compile before you run: `EXPLAIN`
**What it does:** `graph.explain(query)` asks Neo4j to plan the query without executing it.
**Why:** syntax errors and unknown functions are caught cheaply, and the **exact error message** can be sent back to the LLM (layer 4).

In [7]:
for q in ["MATCH (c:Candidate) RETURN c.name LIMIT 3",
          "MATCH (c:Candidate) RETURN c.name GROUP BY c.name",      # SQL habit
          "MATCH (c:Candidate RETURN c"]:
    try:
        graph.explain(q); print("✅ compiles:", q)
    except Exception as exc:
        print("❌", q, "\n   ->", str(exc).splitlines()[0][:150])

✅ compiles: MATCH (c:Candidate) RETURN c.name LIMIT 3
❌ MATCH (c:Candidate) RETURN c.name GROUP BY c.name 
   -> {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'GROUP': expected
❌ MATCH (c:Candidate RETURN c 
   -> {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'RETURN': expected ")", "WHERE", "{" or a parameter (line 1, column 28 (of


## Layer 2: Entity linking

### Step 5: Link the words of the question to graph entities
**What it does:** `EntityLinker` loads the graph's own vocabulary once (skill aliases, candidate names + home city, job titles, organizations, cities) and, for a question, returns **hints** such as
* `'ml' refers to the skill Machine Learning (skillId S05).`
* `'Amina Idrissi' is AMBIGUOUS … C07 (lives in Tangier); C01 (lives in Rabat).`

Matching uses **word boundaries** (so "Java" does not match inside "JavaScript") and **longest alias first** (so "Neo4j Cypher" is the Cypher skill, not Neo4j).

**Why:** this moves the hardest part of the problem (what does the user *mean*?) out of the LLM and into deterministic code that uses real data. It is the graph equivalent of *schema linking* in Text-to-SQL research.

In [8]:
class EntityLinker:
    """Find skills, candidates, jobs, organizations, cities and IDs mentioned in a question,
    using the graph's own vocabulary, and turn them into hints for the LLM."""

    def __init__(self, graph):
        q = graph.run_read
        self.aliases = {r["key"]: (r["skillId"], r["skill"]) for r in q(
            "MATCH (a:Alias)-[:ALIAS_OF]->(s:Skill) RETURN a.key AS key, s.skillId AS skillId, s.name AS skill")}
        self.candidates = q("MATCH (c:Candidate)-[:LIVES_IN]->(ci:City) "
                            "RETURN c.candidateId AS id, c.name AS name, ci.name AS city")
        self.jobs = q("MATCH (j:Job) RETURN j.jobId AS id, j.title AS name, j.status AS status")
        self.orgs = q("MATCH (o:Organization) RETURN o.orgId AS id, o.name AS name")
        self.cities = q("MATCH (ci:City) RETURN ci.cityId AS id, ci.name AS name")

    @staticmethod
    def _find(term: str, text: str) -> bool:
        return re.search(r"(?<![\w])" + re.escape(term) + r"(?![\w])", text, flags=re.IGNORECASE) is not None

    def link(self, question: str) -> list[str]:
        hints = []
        # skills: longest alias first, so 'Neo4j Cypher' wins over 'Neo4j'
        taken = question
        for key in sorted(self.aliases, key=len, reverse=True):
            if self._find(key, taken):
                sid, name = self.aliases[key]
                hints.append(f"'{key}' refers to the skill {name} (skillId {sid}).")
                taken = re.sub(re.escape(key), " ", taken, flags=re.IGNORECASE)
        # people (may be ambiguous!)
        by_name: dict[str, list[dict]] = {}
        for c in self.candidates:
            if self._find(c["name"], question):
                by_name.setdefault(c["name"], []).append(c)
        for name, people in by_name.items():
            if len(people) == 1:
                hints.append(f"'{name}' is candidate {people[0]['id']}.")
            else:
                options = "; ".join(f"{p['id']} (lives in {p['city']})" for p in people)
                hints.append(f"'{name}' is AMBIGUOUS, several candidates have this name: {options}. "
                             "Use other details in the question to choose, otherwise return all of them.")
        for kind, rows in (("job", self.jobs), ("organization", self.orgs), ("city", self.cities)):
            for r in rows:
                if self._find(r["name"], question):
                    extra = f", status {r['status']}" if kind == "job" else ""
                    hints.append(f"'{r['name']}' is the {kind} {r['id']}{extra}.")
        for token in re.findall(r"\b([CJSOEP]\d{2}|AEV\d{2})\b", question):
            hints.append(f"'{token}' is an identifier: use it with the matching *Id property.")
        return hints

def hints_to_text(hints: list[str]) -> str:
    return "Entities found in the question:\n- " + "\n- ".join(hints) if hints else ""

### Step 6: Try the linker
**What to observe:** the alias, the trap (Java vs JavaScript, Spark vs SPARQL) and the ambiguity warning. No LLM is involved in this cell.

In [9]:
linker = EntityLinker(graph)
for q in ["Which candidates claim ML?",
          "Which candidates claim Java but not JavaScript?",
          "Who knows Spark or SPARQL?",
          "Which skills does Amina Idrissi claim?",
          "Show Youssef Karim's events for the Graph Data Engineer job",
          "Does C08 meet the requirements of J01?"]:
    print(q)
    for h in linker.link(q):
        print("   •", h)

Which candidates claim ML?
   • 'ml' refers to the skill Machine Learning (skillId S05).
Which candidates claim Java but not JavaScript?
   • 'javascript' refers to the skill JavaScript (skillId S08).
   • 'java' refers to the skill Java (skillId S07).
Who knows Spark or SPARQL?
   • 'sparql' refers to the skill SPARQL (skillId S11).
   • 'spark' refers to the skill Apache Spark (skillId S10).
Which skills does Amina Idrissi claim?
   • 'Amina Idrissi' is AMBIGUOUS, several candidates have this name: C01 (lives in Rabat); C07 (lives in Tangier). Use other details in the question to choose, otherwise return all of them.
Show Youssef Karim's events for the Graph Data Engineer job
   • 'Youssef Karim' is candidate C02.
   • 'Graph Data Engineer' is the job J01, status OPEN.
Does C08 meet the requirements of J01?
   • 'C08' is an identifier: use it with the matching *Id property.
   • 'J01' is an identifier: use it with the matching *Id property.


## Layer 3: Few-shot examples

### Step 7: The example bank
**What it does:** loads `examples.json`: 14 hand-written question → Cypher pairs that demonstrate the *patterns* of this graph (claim path, alias lookup, "meets all mandatory requirements", location rule, ordered history, assessments…).

⚠️ **Methodology:** these examples are **different** from the questions of the benchmark used in notebook 04 (`benchmark.json`). If the same questions were in both, we would measure memorisation, not generalisation. Some share a *template* (e.g. "meets all requirements of J06" vs "of J01") — discuss with students whether this is fair (it is realistic: a production bank contains templates).

In [10]:
from t2c import load_json
examples = load_json("examples.json")
pd.DataFrame(examples)[["id", "question"]]

,id,question
0,X01,List the closed jobs.
1,X02,Which candidates live in Fez?
2,X03,"Which skills does candidate C02 claim, with level and verification status?"
3,X04,Which candidates claim K8s?
4,X05,"What are all the requirements of job J05, mandatory and preferred?"
5,X06,Which candidates meet all mandatory skill requirements of job J06?
6,X07,"Where has candidate C04 worked, and when?"
7,X08,"Show the full application history of candidate C09 for job J05, in order."
8,X09,How many skill claims does each candidate have?
9,X10,Which candidates are available onsite in the city of job J06?


### Step 8: Choosing the most relevant examples
**What it does:** `ExampleSelector` ranks examples by similarity to the question and keeps the top *k*:
* with an **embedding model** (`"qwen3-embedding:4b` via Ollama): cosine similarity between sentence vectors — captures meaning ("Who knows ML?" ≈ "Which candidates claim K8s?");
* without embeddings: Jaccard overlap of words — a cheap fallback.

`examples_to_messages` turns examples into fake previous turns (user question → assistant Cypher). Models imitate the *format* and the *patterns* of previous turns very strongly.

**Why not put all 14 examples in the prompt?** Longer prompts are slower on a laptop and irrelevant examples can mislead small models. 3–5 relevant examples is the usual sweet spot.

In [11]:
def _tokens(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na, nb = math.sqrt(sum(x * x for x in a)), math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0

class ExampleSelector:
    """Pick the k examples most similar to the question (embeddings if available, else word overlap)."""

    def __init__(self, examples: list[dict], embed_fn=None):
        self.examples = examples
        self.embed_fn = embed_fn
        self.vectors = embed_fn([e["question"] for e in examples]) if embed_fn else None

    def select(self, question: str, k: int = 4) -> list[dict]:
        if self.embed_fn:
            qv = self.embed_fn([question])[0]
            scores = [cosine(qv, v) for v in self.vectors]
        else:
            qt = _tokens(question)
            scores = [len(qt & _tokens(e["question"])) / len(qt | _tokens(e["question"])) for e in self.examples]
        ranked = sorted(zip(scores, self.examples), key=lambda p: p[0], reverse=True)
        return [e for _, e in ranked[:k]]

def examples_to_messages(examples: list[dict]) -> list[dict]:
    """Few-shot as fake previous turns: user question -> assistant cypher."""
    messages = []
    for ex in examples:
        messages.append({"role": "user", "content": f"Question: {ex['question']}"})
        messages.append({"role": "assistant", "content": f"```cypher\n{ex['cypher']}\n```"})
    return messages

### Step 9 — Lexical vs. semantic selection
**What it does:** compares the examples chosen by each method for the same question. If the embedding model is not installed, run `ollama pull "qwen3-embedding:4b` (≈ 270 MB) or keep the lexical selector.

In [12]:
lexical = ExampleSelector(examples)
try:
    semantic = ExampleSelector(examples, embed_fn=lambda texts: llm.embed(texts, model=EMBED_MODEL))
except Exception as exc:
    print("Embeddings unavailable, using lexical selection only:", exc)
    semantic = lexical

q = "Who is qualified for every compulsory skill of the Platform Engineer job?"
print("Lexical :", [e["id"] + " " + e["question"] for e in lexical.select(q, 3)])
print("Semantic:", [e["id"] + " " + e["question"] for e in semantic.select(q, 3)])

Lexical : ['X08 Show the full application history of candidate C09 for job J05, in order.', 'X06 Which candidates meet all mandatory skill requirements of job J06?', 'X05 What are all the requirements of job J05, mandatory and preferred?']
Semantic: ['X06 Which candidates meet all mandatory skill requirements of job J06?', 'X03 Which skills does candidate C02 claim, with level and verification status?', 'X09 How many skill claims does each candidate have?']


## Layer 4: Self-correction and the full pipeline

### Step 10; The `Text2Cypher` class
**What it does:** `build_messages` assembles *system prompt → selected examples → (entity hints + question)*. `__call__` then loops up to `max_retries + 1` times:
1. ask the LLM, 2. extract the query, 3. guard, 4. `EXPLAIN`, 5. run in a read transaction.

If any step raises an error, the model's answer **and the error message** are appended to the conversation with "Fix the query", and we try again. Every attempt is recorded in `trace`, which is what you show students when explaining *why* a result came out.

**Limitation to discuss:** the loop only fixes errors that *raise* something. A query that runs and returns wrong rows is not retried (how could we detect it? → notebook 04).

In [13]:
class Text2Cypher:
    """question -> (entity hints + few-shot) -> LLM -> extract -> guard -> EXPLAIN -> run,
    retrying with the error message when something fails."""

    def __init__(self, graph, llm, system_prompt: str, linker: EntityLinker | None = None,
                 selector: ExampleSelector | None = None, k: int = 4, max_retries: int = 2):
        self.graph, self.llm, self.system_prompt = graph, llm, system_prompt
        self.linker, self.selector, self.k, self.max_retries = linker, selector, k, max_retries

    def build_messages(self, question: str) -> list[dict]:
        messages = [{"role": "system", "content": self.system_prompt}]
        if self.selector:
            messages += examples_to_messages(self.selector.select(question, self.k))
        hint = hints_to_text(self.linker.link(question)) if self.linker else ""
        user = f"{hint}\n\nQuestion: {question}" if hint else f"Question: {question}"
        messages.append({"role": "user", "content": user})
        return messages

    def __call__(self, question: str) -> dict:
        start = time.time()
        messages = self.build_messages(question)
        trace, cypher, rows, error = [], None, None, None
        for attempt in range(1, self.max_retries + 2):
            raw = self.llm(messages)
            cypher = extract_cypher(raw)
            try:
                check_read_only(cypher)
                self.graph.explain(cypher)
                rows = self.graph.run_read(cypher)
                error = None
                trace.append({"attempt": attempt, "cypher": cypher, "status": "ok", "rows": len(rows)})
                break
            except Exception as exc:  # noqa: BLE001  (we want every failure to be fed back)
                error = f"{type(exc).__name__}: {exc}"
                trace.append({"attempt": attempt, "cypher": cypher, "status": "error", "error": error})
                messages += [
                    {"role": "assistant", "content": raw},
                    {"role": "user", "content": f"That query failed with this error:\n{error[:800]}\n"
                                                "Fix the query. Answer with the corrected Cypher only."}]
        return {"question": question, "cypher": cypher, "rows": rows, "error": error,
                "attempts": len(trace), "trace": trace, "seconds": round(time.time() - start, 2)}

### Step 11: Build the pipeline and a display helper

In [14]:
t2c_pipeline = Text2Cypher(graph, llm, system_prompt, linker=linker, selector=semantic, k=4, max_retries=2)

def show(result):
    for step in result["trace"]:
        flag = "✅" if step["status"] == "ok" else "❌"
        print(f"{flag} attempt {step['attempt']}:\n{step['cypher']}")
        if step["status"] != "ok":
            print("   error:", step["error"][:300])
    print(f"({result['seconds']}s)")
    if result["rows"] is not None:
        display(pd.DataFrame(result["rows"]))

### Step 12: Re-run the hard questions of notebook 02
**Expected answers:** ML → C03 · Java → C04, C10 · J01 → C01 only · Amina Idrissi in Tangier → SPARQL, RDF, TypeScript.
Compare with what you got zero-shot. Look at the prompt with `t2c_pipeline.build_messages(q)` to see exactly what the model received.

In [15]:
for q in ["Which candidates claim ML?",
          "Which candidates claim Java?",
          "Which candidates meet all mandatory skill requirements of job J01?",
          "Which skills does the Amina Idrissi who lives in Tangier claim?"]:
    print("\nQ:", q)
    show(t2c_pipeline(q))


Q: Which candidates claim ML?
✅ attempt 1:
MATCH (:Alias {key: toLower('ml')})-[:ALIAS_OF]->(s:Skill {name: 'Machine Learning'})
MATCH (s)<-[:ABOUT_SKILL]-(cl:SkillClaim)<-[:MAKES_CLAIM]-(c:Candidate)
RETURN c.candidateId AS candidateId, c.name AS name, cl.level AS level, cl.verificationStatus AS verificationStatus
(13.8s)


,candidateId,name,level,verificationStatus
0,C03,Sara Haddad,3,VERIFIED



Q: Which candidates claim Java?
✅ attempt 1:
MATCH (:Alias {key: toLower('java')})-[:ALIAS_OF]->(s:Skill)<-[:ABOUT_SKILL]-(cl:SkillClaim)<-[:MAKES_CLAIM]-(c:Candidate)
RETURN c.candidateId AS candidateId, c.name AS name, cl.level AS level, cl.verificationStatus AS verificationStatus
(12.43s)


,candidateId,name,level,verificationStatus
0,C04,Omar Naji,3,VERIFIED
1,C10,Hamza Tazi,3,VERIFIED



Q: Which candidates meet all mandatory skill requirements of job J01?
✅ attempt 1:
MATCH (j:Job {jobId: 'J01'})-[:REQUIRES_SKILL {priority: 'MANDATORY'}]->(ms:Skill)
WITH j, count(ms) AS nMand
MATCH (j)-[r:REQUIRES_SKILL {priority: 'MANDATORY'}]->(s:Skill)<-[:ABOUT_SKILL]-(cl:SkillClaim {verificationStatus: 'VERIFIED'})<-[:MAKES_CLAIM]-(c:Candidate)
WHERE cl.level >= r.minLevel
WITH nMand, c, count(DISTINCT s) AS nMet
WHERE nMet = nMand
RETURN c.candidateId AS candidateId, c.name AS name
(4.01s)


,candidateId,name
0,C01,Amina Idrissi



Q: Which skills does the Amina Idrissi who lives in Tangier claim?
❌ attempt 1:
MATCH (c:Candidate {name: 'Amina Idrissi', LIVES_IN: (l:City {name: 'Tangier'})})-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill)
RETURN s.name AS skill, cl.level AS level, cl.verificationStatus AS verificationStatus
   error: CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '{': expected
  "!="
  "%"
  ")"
  "*"
  "+"
  "-"
  "/"
  "::"
  "<"
  "<="
  "<>"
  "="
  "=~"
  ">"
  ">="
  "AND"
  "CONTAINS"
  "ENDS"
  "IN"
  "IS"
  "OR"
  "STARTS"
  "XOR"
 
✅ attempt 2:
MATCH (c:Candidate {name: 'Amina Idrissi'})-[:LIVES_IN]->(l:City {name: 'Tangier'})-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill)
RETURN s.name AS skill, cl.level AS level, cl.verificationStatus AS verificationStatus
(4.31s)


""


### Step 13: Watching self-correction happen (deterministic demo)
Real models do not fail on demand, so we wrap the LLM in a small "saboteur" that returns a broken query on the **first** call only. The trace shows: attempt 1 fails at `EXPLAIN`, the error is sent back, attempt 2 succeeds.
This is a good moment to show the conversation that was actually sent (`messages`) on the second attempt.

In [16]:
class BrokenFirstAnswer:
    """Wraps a real LLM; the first call returns an invalid query."""
    def __init__(self, inner):
        self.inner, self.calls = inner, 0
    def __call__(self, messages, fmt=None):
        self.calls += 1
        if self.calls == 1:
            return "```cypher\nMATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim RETURN c.name\n```"
        return self.inner(messages, fmt)

demo = Text2Cypher(graph, BrokenFirstAnswer(llm), system_prompt, linker=linker, selector=semantic)
show(demo("Which candidates claim Docker, and is the claim verified?"))

❌ attempt 1:
MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim RETURN c.name
   error: CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'RETURN': expected ")", "WHERE", "{" or a parameter (line 1, column 60 (offset: 59))
"EXPLAIN MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim RETURN c.name"
                                           
✅ attempt 2:
MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(:Skill {name: 'Docker'})
RETURN c.candidateId AS candidateId, c.name AS name, cl.level AS level, cl.verificationStatus AS verificationStatus
(4.28s)


,candidateId,name,level,verificationStatus
0,C03,Sara Haddad,2,UNVERIFIED
1,C01,Amina Idrissi,1,UNVERIFIED
2,C10,Hamza Tazi,3,VERIFIED
3,C02,Youssef Karim,3,VERIFIED


### Step 14 (optional): Structured output with a JSON schema
**What it does:** recent Ollama versions accept `format=<JSON schema>` and force the model to answer with JSON that matches it. We ask for `{"reasoning": ..., "cypher": ...}`.
**Why:** no more fragile parsing of code fences, and the `reasoning` field (written *before* the query) acts as a lightweight chain-of-thought. Trade-off: some small models produce worse Cypher when constrained to JSON — measure it in notebook 04!

In [17]:
answer_schema = {
    "type": "object",
    "properties": {"reasoning": {"type": "string"}, "cypher": {"type": "string"}},
    "required": ["reasoning", "cypher"],
}
messages = t2c_pipeline.build_messages("Which open jobs are hybrid, and in which city?")
messages[0]["content"] += "\nAnswer as JSON with the keys 'reasoning' (one short paragraph) and 'cypher'."
try:
    out = json.loads(llm(messages, fmt=answer_schema))
    print("Reasoning:", out["reasoning"], "\n\nCypher:\n", out["cypher"])
    display(pd.DataFrame(graph.run_read(check_read_only(out["cypher"]))))
except Exception as exc:
    print("Structured output failed (old Ollama version or model issue):", exc)

Reasoning: We need to find open jobs with a workMode of 'HYBRID' and return their jobId, title, and the city they are based in. 

Cypher:
 MATCH (j:Job {status: 'OPEN', workMode: 'HYBRID'})-[:BASED_IN]->(city:City) RETURN j.jobId AS jobId, j.title AS title, city.name AS city


,jobId,title,city
0,J01,Graph Data Engineer,Rabat
1,J05,Big Data Engineer,Casablanca


## Exercises
1. Add **organization aliases** ("NorthStar" → NorthStar Analytics) to the linker. Where should these aliases live: in Python, or as `Alias` nodes in the graph?
2. When the linker finds an ambiguous name and the question gives no extra detail, should the system (a) return both people, (b) ask the user a clarification question? Implement (b).
3. Extend the retry loop: if the query returns **0 rows**, ask the model once "The query returned no rows. If you think that is correct, return the same query; otherwise fix it." Measure in notebook 04 whether it helps or hurts.
4. Write 3 new examples for patterns the model still gets wrong and check that the selector picks them.